# 03B — Calibration interne et sélection des modèles SIMCA

Ce notebook utilise exclusivement les lots autorisés pour la calibration
interne. Il construit des prédictions out-of-fold groupées par objet, évalue
les huit tracks E1–E8 et calibre les règles de décision 2-way et 3-way.

Les seuils et modèles sont sélectionnés sans score composite. Les contraintes
de sécurité sur les faux négatifs sont appliquées avant celles portant sur les
faux positifs. Les politiques 3-way sont calibrées par cross-fitting afin que
le fold évalué ne participe pas à l'estimation de ses seuils.

Pour chaque étape, le notebook conserve les candidats évalués, les candidats
éliminés, la raison de l'élimination, la valeur observée et la référence
utilisée. Le front de Pareto n'est calculé qu'après les contraintes de risque,
la vérification de complétude et la réduction de complexité.

Les lots de validation et de test externes ne sont jamais utilisés dans ce
notebook.

## 1 — Gouvernance, version et artefacts verrouillés


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyarrow.parquet as pq

from IPython.display import display

current_dir = Path.cwd().resolve()
if (current_dir / "src").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise RuntimeError(
        "Launch 03B from the repository or notebooks directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import (
    build_simca_track_contracts,
    sha256_dataframe,
    sha256_file,
    sha256_payload,
    validate_selection_only_protocol_lineage,
    verify_frozen_protocol,
)
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
)
from src.utils import save_parquet
from src.workflows.matrix_preprocessing import (
    assert_wavelength_lock,
    build_wavelength_config,
)
from src.workflows.pca_selection import (
    hash_pca_input_artifacts,
    hash_pca_review_table,
    pca_input_fingerprint,
    validate_pca_preprocessing_shortlist,
)
from src.workflows.protocol_audit import (
    assert_no_forbidden_score_columns,
)
from src.workflows.protocol_split import eligible_object_ids
from src.workflows.simca_calibration_registry import (
    build_internal_calibration_candidate_runs,
)
from src.workflows.simca_calibration_selection import (
    build_model_metrics,
    finalize_streamed_selection_audit,
    reduce_threshold_policies_from_checkpoint_8tracks,
    sample_threshold_candidates_for_plot,
    select_threshold_policies_from_candidate_cache_8tracks,
    select_calibrated_models,
    summarize_selection_audit,
)
from src.workflows.simca_internal_calibration import (
    build_calibration_folds,
    build_reference_object_table,
    load_selected_oof_predictions_from_checkpoint_8tracks,
    resolve_internal_calibration_checkpoint_run_8tracks,
    run_internal_calibration_8tracks,
)
from src.visualization.plot_model_selection import(
    plot_threshold_tradeoff,
)

In [2]:
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_"
    f"{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)

protocol_dir = PROJECT_ROOT.joinpath(
    *expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR
)
qc_dir = PROJECT_ROOT.joinpath(
    *expcfg.QC_RESULTS_RELATIVE_DIR
)
matrix_dir = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.MATRIX_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
pca_dir = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.PCA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
output_dir = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
)
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_dir = (
    output_dir / expcfg.INTERNAL_CALIBRATION_CHECKPOINT_DIRNAME
    if expcfg.INTERNAL_CALIBRATION_CHECKPOINT_ENABLED
    else None
)

output_paths = {
    key: output_dir / filename
    for key, filename
    in expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES.items()
}
threshold_candidate_cache_path = (
    output_dir
    / expcfg.INTERNAL_CALIBRATION_THRESHOLD_CANDIDATE_CACHE_FILENAME
)

verify_frozen_protocol(protocol_dir, strict=True)

protocol_lock = json.loads(
    (
        protocol_dir
        / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]
    ).read_text(encoding="utf-8")
)
protocol_hash = str(protocol_lock["lock_sha256"])

if tuple(expcfg.INTERNAL_CALIBRATION_OBJECT_THRESHOLDS) != (
    0.75,
    0.80,
):
    raise RuntimeError(
        "03B expects the frozen object thresholds (0.75, 0.80)."
    )

In [3]:
split_manifest = pd.read_parquet(
    qc_dir / expcfg.QC_OUTPUT_FILENAMES["split_manifest"]
)
wavelength_lock = pd.read_parquet(
    matrix_dir
    / expcfg.MATRIX_OUTPUT_FILENAMES["wavelength_config"]
)
pca_selected = pd.read_parquet(
    pca_dir / expcfg.PCA_OUTPUT_FILENAMES["selected"]
)
pca_review = pd.read_parquet(
    pca_dir / expcfg.PCA_OUTPUT_FILENAMES["artifact_review"]
)

pca_input_hash = pca_input_fingerprint(
    hash_pca_input_artifacts(
        PROJECT_ROOT,
        results_tag=RESULTS_TAG,
    )
)
pca_review_hash = hash_pca_review_table(pca_review)

pca_protocol_hashes = (
    pca_selected["protocol_hash"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)
if len(pca_protocol_hashes) != 1:
    raise RuntimeError(
        "PCA shortlist must have one execution protocol hash."
    )
execution_protocol_hash = pca_protocol_hashes[0]
expected_parent_protocol_hash = (
    expcfg.INTERNAL_CALIBRATION_SELECTION_PARENT_PROTOCOL_HASH
)
if execution_protocol_hash != expected_parent_protocol_hash:
    raise RuntimeError(
        "PCA execution protocol does not match the declared "
        "selection-amendment parent."
    )
validate_pca_preprocessing_shortlist(
    pca_selected,
    max_per_family=(
        expcfg.MAX_PCA_PREPROCESSINGS_PER_FAMILY
    ),
    expected_families=(
        expcfg.PCA_SELECTION_EXPECTED_FAMILIES
    ),
    expected_protocol_hash=execution_protocol_hash,
    expected_input_fingerprint=pca_input_hash,
    expected_review_hash=pca_review_hash,
)

pca_selection_fingerprint = sha256_dataframe(pca_selected)

object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(
        *expcfg.DATABASE_H5_RELATIVE_PATH
    ),
    reconstruct_heavy_object_arrays=True,
)

if expcfg.USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = (
        select_wavelength_range_from_database(
            object_db=object_db,
            image_db=image_db,
            min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
            max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
        )
    )
else:
    wavelengths = np.asarray(
        next(iter(object_db.values()))["wavelengths"]
    )

wavelength_candidate = build_wavelength_config(
    image_db,
    object_db,
    wavelength_mode=expcfg.DEFAULT_WAVELENGTH_MODE,
    protocol_version=expcfg.PROTOCOL_VERSION,
    n_remove_start=expcfg.N_REMOVE_START,
    n_stop_end=expcfg.N_STOP_END,
    window_min_nm=(
        expcfg.WAVELENGTH_WINDOW_MIN_NM
        if expcfg.USE_WAVELENGTH_WINDOW
        else None
    ),
    window_max_nm=(
        expcfg.WAVELENGTH_WINDOW_MAX_NM
        if expcfg.USE_WAVELENGTH_WINDOW
        else None
    ),
)
assert_wavelength_lock(
    wavelength_lock,
    wavelength_candidate,
)

## 2 — Contrats des huit tracks


In [4]:
track_contracts = build_simca_track_contracts()

expected_track_ids = {
    f"E{index}" for index in range(1, 9)
}
if (
    len(track_contracts) != 8
    or set(track_contracts["track_id"]) != expected_track_ids
):
    raise RuntimeError(
        "The track contract must contain exactly E1-E8."
    )

if set(expcfg.INTERNAL_CALIBRATION_ALLOWED_UNSUPPORTED_TRACK_IDS):
    raise RuntimeError(
        "No unsupported track is allowed in the v5 protocol."
    )

save_parquet(
    track_contracts,
    output_paths["track_contracts"],
)
track_contract_hash = sha256_dataframe(track_contracts)

display(
    track_contracts[
        [
            "track_id",
            "evaluation_track",
            "training_matrix_family",
            "projection_level",
            "decision_mode",
            "primary_unit",
        ]
    ]
)

,track_id,evaluation_track,training_matrix_family,projection_level,decision_mode,primary_unit
0,E1,object_train__object_projection__2way,object_matrix,object_projection,2way,object
1,E2,object_train__object_projection__3way,object_matrix,object_projection,3way,object
2,E3,object_train__pixel_projection__2way,object_matrix,pixel_projection,2way,source_image
3,E4,object_train__pixel_projection__3way,object_matrix,pixel_projection,3way,source_image
4,E5,pixel_train__object_projection__2way,pixel_matrix,object_projection,2way,object
5,E6,pixel_train__object_projection__3way,pixel_matrix,object_projection,3way,object
6,E7,pixel_train__pixel_projection__2way,pixel_matrix,pixel_projection,2way,source_image
7,E8,pixel_train__pixel_projection__3way,pixel_matrix,pixel_projection,3way,source_image


## 3 — Références QC-éligibles et folds groupés


In [5]:
calibration_ids = eligible_object_ids(
    split_manifest,
    "calibration",
)

reference_objects = build_reference_object_table(
    object_db,
    allowed_object_ids=calibration_ids,
    batches=expcfg.INTERNAL_CALIBRATION_BATCHES,
    classes=expcfg.REFERENCE_CLASSES,
)

observed_batches = set(
    pd.to_numeric(
        reference_objects["batch"],
        errors="raise",
    ).astype(int)
)
if observed_batches & set(
    expcfg.INTERNAL_CALIBRATION_FORBIDDEN_BATCHES
):
    raise RuntimeError(
        "A forbidden batch entered the calibration reference."
    )

calibration_folds, fold_diagnostics = (
    build_calibration_folds(
        reference_objects,
        n_splits=expcfg.INTERNAL_CALIBRATION_N_SPLITS,
        n_size_bins=(
            expcfg.INTERNAL_CALIBRATION_SIZE_N_BINS
        ),
        random_state=(
            expcfg.INTERNAL_CALIBRATION_FOLD_RANDOM_STATE
        ),
        require_complete_coverage=True,
    )
)

if set(
    calibration_folds["object_id"].astype(str)
) - set(map(str, calibration_ids)):
    raise RuntimeError(
        "An object outside the QC calibration split entered 03B."
    )

save_parquet(
    calibration_folds,
    output_paths["folds"],
)
save_parquet(
    fold_diagnostics,
    output_paths["fold_diagnostics"],
)

fold_contract_hash = sha256_dataframe(
    calibration_folds
)
display(fold_diagnostics)

,fold_id,n_groups,n_objects,n_target_objects,n_non_target_objects,n_batch_1_objects,n_batch_2_objects,median_object_size,coverage_complete
0,0,2,104,52,52,52,52,75.5,True
1,1,2,105,46,59,46,59,65.0,True


## 4 — Catalogue scientifique et exécutions

`model_id` identifie les paramètres scientifiques d’un modèle. Les graines
sont enregistrées dans `candidate_runs`, mais ne créent pas de nouveaux modèles.
`fit_id` et `projection_id` sont des identifiants techniques permettant de
réutiliser les calculs coûteux.

In [6]:
execution_configurations = (
    build_internal_calibration_candidate_runs(
        pca_selected,
        track_contracts,
        matrix_methods=(
            expcfg.INTERNAL_CALIBRATION_MATRIX_METHODS
        ),
        m_values=expcfg.INTERNAL_CALIBRATION_M_VALUES,
        pixel_strategies=(
            expcfg.INTERNAL_CALIBRATION_PIXEL_STRATEGIES
        ),
        n_components_values=(
            expcfg.INTERNAL_CALIBRATION_N_COMPONENTS_VALUES
        ),
        rule_variants=(
            expcfg.INTERNAL_CALIBRATION_RULE_VARIANTS
        ),
        alpha_values=(
            expcfg.INTERNAL_CALIBRATION_ALPHA_VALUES
        ),
        sg_windows=(
            expcfg.INTERNAL_CALIBRATION_SG_WINDOWS
        ),
        sg_polyorders=(
            expcfg.INTERNAL_CALIBRATION_SG_POLYORDERS
        ),
        dilation_radii=(
            expcfg.INTERNAL_CALIBRATION_DILATION_RADII
        ),
        random_seeds=(
            expcfg.INTERNAL_CALIBRATION_RANDOM_SEEDS
        ),
    )
)

model_catalog = (
    execution_configurations[
        list(expcfg.INTERNAL_CALIBRATION_MODEL_CATALOG_COLUMNS)
    ]
    .drop_duplicates("model_id")
    .reset_index(drop=True)
)

candidate_runs = (
    execution_configurations[
        list(expcfg.INTERNAL_CALIBRATION_CANDIDATE_RUN_COLUMNS)
    ]
    .drop_duplicates(["model_id", "random_state"])
    .reset_index(drop=True)
)

if candidate_runs.duplicated(
    ["model_id", "random_state"]
).any():
    raise RuntimeError(
        "(model_id, random_state) is not unique."
    )

model_parameter_identity = (
    model_catalog.groupby("model_id", dropna=False)[
        list(expcfg.SIMCA_MODEL_PARAMETER_COLUMNS)
    ]
    .nunique(dropna=False)
    .max(axis=1)
)
if model_parameter_identity.gt(1).any():
    raise RuntimeError(
        "A model_id maps to multiple parameter sets."
    )

track_coverage = (
    model_catalog.groupby(
        ["track_id", "evaluation_track"],
        as_index=False,
    )
    .agg(n_models=("model_id", "nunique"))
)

if set(track_coverage["track_id"]) != expected_track_ids:
    raise RuntimeError(
        "At least one track has no initial candidate."
    )

save_parquet(
    model_catalog,
    output_paths["model_catalog"],
)
save_parquet(
    candidate_runs,
    output_paths["candidate_runs"],
)

configuration_hash = sha256_dataframe(
    execution_configurations
)

fig = px.bar(
    track_coverage,
    x="track_id",
    y="n_models",
    hover_data=["evaluation_track"],
    title="Candidats scientifiques initiaux par track",
    labels={
        "track_id": "Track",
        "n_models": "Nombre de modèles",
    },
)
fig.update_layout(
    template="plotly_white",
    showlegend=False,
)
fig.show()

## 5 — Exécution OOF avec fits et projections partagés

Les grandes prédictions candidates restent dans les shards de checkpoint.
Les tables compactes de diagnostic et de calibration sont conservées en mémoire.

Le protocole courant porte l'amendement de sélection, tandis que le hash
parent reste attaché aux PCA et aux fits. Avant le runner, les empreintes
PCA, tracks, folds et configurations doivent toutes correspondre exactement
au checkpoint parent complet; sinon l'exécution est bloquée.

In [7]:
if not expcfg.INTERNAL_CALIBRATION_RUN:
    raise RuntimeError(
        "Enable INTERNAL_CALIBRATION_RUN in experiment_config.py."
    )

if checkpoint_dir is None:
    raise RuntimeError(
        "Selection-only amendment reuse requires checkpoints."
    )

checkpoint_execution_context = {
    "protocol_hash": execution_protocol_hash,
    "pca_selection_fingerprint": (
        pca_selection_fingerprint
    ),
    "track_contract_hash": track_contract_hash,
    "fold_contract_hash": fold_contract_hash,
    "configuration_hash": configuration_hash,
}
expected_fit_ids = (
    execution_configurations["fit_id"]
    .drop_duplicates()
    .astype(str)
    .tolist()
)
resolved_checkpoint_run_dir = (
    resolve_internal_calibration_checkpoint_run_8tracks(
        checkpoint_dir,
        checkpoint_context=checkpoint_execution_context,
        expected_fit_config_ids=expected_fit_ids,
    )
)
parent_checkpoint_manifest = json.loads(
    (resolved_checkpoint_run_dir / "manifest.json").read_text(
        encoding="utf-8"
    )
)
protocol_lineage_checks = (
    validate_selection_only_protocol_lineage(
        current_protocol_hash=protocol_hash,
        execution_protocol_hash=execution_protocol_hash,
        expected_parent_protocol_hash=(
            expected_parent_protocol_hash
        ),
        amendment_scope=(
            expcfg.INTERNAL_CALIBRATION_SELECTION_AMENDMENT_SCOPE
        ),
        selection_profile_id=(
            expcfg.INTERNAL_CALIBRATION_SELECTION_PROFILE_ID
        ),
        selection_parent_profile_id=(
            expcfg.INTERNAL_CALIBRATION_SELECTION_PARENT_PROFILE_ID
        ),
        checkpoint_manifest=parent_checkpoint_manifest,
        expected_execution_context=checkpoint_execution_context,
        strict=True,
    )
)
display(protocol_lineage_checks)
print(
    "Validated parent checkpoint: "
    f"{resolved_checkpoint_run_dir.name}"
)

calibration_results = run_internal_calibration_8tracks(
    object_db=object_db,
    folds=calibration_folds,
    configurations=execution_configurations,
    wavelengths=wavelengths,
    under_m_policy=(
        expcfg.INTERNAL_CALIBRATION_UNDER_M_POLICY
    ),
    verbose=expcfg.INTERNAL_CALIBRATION_VERBOSE,
    checkpoint_dir=checkpoint_dir,
    checkpoint_context=checkpoint_execution_context,
    resume_from_checkpoint=(
        expcfg.INTERNAL_CALIBRATION_RESUME_FROM_CHECKPOINT
    ),
    keep_oof_in_memory=False,
    keep_threshold_metrics_in_memory=False,
)

checkpoint_run_dir = calibration_results[
    "checkpoint_run_dir"
]
if Path(checkpoint_run_dir).resolve() != (
    resolved_checkpoint_run_dir.resolve()
):
    raise RuntimeError(
        "Runner did not reuse the validated parent checkpoint."
    )
fit_diagnostics = calibration_results[
    "fit_diagnostics"
]
rule_diagnostics = calibration_results[
    "rule_diagnostics"
]
projection_shift = calibration_results[
    "projection_shift"
]
technical_events = calibration_results[
    "technical_events"
]

for table, key in (
    (fit_diagnostics, "fit_diagnostics"),
    (rule_diagnostics, "rule_diagnostics"),
    (projection_shift, "projection_shift"),
    (technical_events, "technical_events"),
):
    save_parquet(table, output_paths[key])

display(
    technical_events.groupby(
        ["stage", "status", "reason_code"],
        dropna=False,
        as_index=False,
    ).size()
)

,check,passed,detail
0,protocol_amendment_scope_is_selection_only,True,scope=selection_only
1,selection_amendment_has_distinct_current_protocol,True,current=af19c5f35d24a81a9c34d0a37c1f2776a31f5a...
2,execution_protocol_matches_declared_parent,True,expected=5d66e659d7da4e69fa647123058bcea08d33f...
3,selection_profile_has_declared_parent,True,profile=fn-priority_v1_scope-aware-amendment-1...
4,checkpoint_protocol_matches_execution_parent,True,expected=5d66e659d7da4e69fa647123058bcea08d33f...
5,checkpoint_protocol_version_matches_current,True,"expected=8tracks_v5, observed=8tracks_v5"
6,checkpoint_schema_version_matches_current,True,"expected=8tracks_v5, observed=8tracks_v5"
7,checkpoint_protocol_hash_matches_current_execu...,True,expected=5d66e659d7da4e69fa647123058bcea08d33f...
8,checkpoint_pca_selection_fingerprint_matches_c...,True,expected=4b619f17e83817ad65791cf1e086fca08296a...
9,checkpoint_track_contract_hash_matches_current...,True,expected=bf9e36e1258de42dfdaf273f8df3e9c3a6dce...


Validated parent checkpoint: run_49306c622682e912469d


,stage,status,reason_code,size


## 6 — Sélection des politiques de seuil

Les contraintes de FN sont appliquées avant les contraintes de FP. Parmi les
politiques admissibles, la sélection est lexicographique : plateau FN, puis
plateau FP, puis incertitude pour les modes 3-way.

Les seuils objets 0.75 et 0.80 restent tous deux candidats pour E3 et E7.

Les métriques d'exactitude ne sont imposées que lorsqu'elles sont identifiables :
projection directe au niveau objet, ou agrégation pixel-vers-objet. E3 utilise
des limites FN amendées par portée; les autres limites restent inchangées.
Les politiques agrégées sont mises en cache avec leur contexte scientifique afin
de pouvoir refaire cette sélection sans recalculer les fits ni les projections.

In [8]:
threshold_audit_work_path = output_paths["selection_audit"].with_name(
    "_threshold_selection_audit.parquet"
)
threshold_candidate_cache_context = {
    "execution_protocol_hash": execution_protocol_hash,
    "pca_selection_fingerprint": pca_selection_fingerprint,
    "track_contract_hash": track_contract_hash,
    "fold_contract_hash": fold_contract_hash,
    "configuration_hash": configuration_hash,
    "checkpoint_run_name": Path(checkpoint_run_dir).name,
    "n_splits": expcfg.INTERNAL_CALIBRATION_N_SPLITS,
    "object_thresholds": list(
        expcfg.INTERNAL_CALIBRATION_OBJECT_THRESHOLDS
    ),
    "direct_2way_threshold": (
        expcfg.INTERNAL_CALIBRATION_DIRECT_2WAY_THRESHOLD
    ),
    "pixel_vote_center": (
        expcfg.INTERNAL_CALIBRATION_PIXEL_VOTE_CENTER
    ),
    "lower_quantiles": list(
        expcfg.INTERNAL_CALIBRATION_THREE_WAY_LOWER_QUANTILES
    ),
    "upper_quantiles": list(
        expcfg.INTERNAL_CALIBRATION_THREE_WAY_UPPER_QUANTILES
    ),
    "threshold_crossfit": (
        expcfg.INTERNAL_CALIBRATION_THRESHOLD_CROSSFIT
    ),
    "target_uncertain_policy": (
        expcfg.INTERNAL_CALIBRATION_TARGET_UNCERTAIN_POLICY
    ),
}
selection_profile_payload = {
    "current_protocol_hash": protocol_hash,
    "execution_parent_protocol_hash": (
        execution_protocol_hash
    ),
    "amendment_scope": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_AMENDMENT_SCOPE
    ),
    "profile_id": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_PROFILE_ID
    ),
    "parent_profile_id": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_PARENT_PROFILE_ID
    ),
    "amendment_reason": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_AMENDMENT_REASON
    ),
    "constraints": (
        expcfg.INTERNAL_CALIBRATION_THRESHOLD_CONSTRAINTS
    ),
    "overrides": (
        expcfg.INTERNAL_CALIBRATION_THRESHOLD_OVERRIDES
    ),
    "accuracy_contexts": (
        expcfg.INTERNAL_CALIBRATION_ACCURACY_CONTEXTS
    ),
    "threshold_priority": (
        expcfg.INTERNAL_CALIBRATION_THRESHOLD_PRIORITY
    ),
    "model_priority": (
        expcfg.INTERNAL_CALIBRATION_MODEL_PRIORITY
    ),
    "pareto_objectives": (
        expcfg.INTERNAL_CALIBRATION_PARETO_OBJECTIVES
    ),
}
selection_profile_hash = sha256_payload(
    selection_profile_payload
)
reuse_threshold_candidate_cache = (
    expcfg.INTERNAL_CALIBRATION_REUSE_THRESHOLD_CANDIDATE_CACHE
    and threshold_candidate_cache_path.exists()
    and not (
        expcfg.INTERNAL_CALIBRATION_REBUILD_THRESHOLD_CANDIDATE_CACHE
    )
)
if reuse_threshold_candidate_cache:
    threshold_reduction = (
        select_threshold_policies_from_candidate_cache_8tracks(
            threshold_candidate_cache_path,
            execution_configurations,
            threshold_metrics_path=(
                output_paths["threshold_metrics"]
            ),
            threshold_audit_output_path=(
                threshold_audit_work_path
            ),
            threshold_candidate_cache_context=(
                threshold_candidate_cache_context
            ),
            verbose=expcfg.INTERNAL_CALIBRATION_VERBOSE,
        )
    )
else:
    threshold_reduction = (
        reduce_threshold_policies_from_checkpoint_8tracks(
            checkpoint_run_dir,
            execution_configurations,
            threshold_metrics_output_path=(
                output_paths["threshold_metrics"]
            ),
            threshold_audit_output_path=(
                threshold_audit_work_path
            ),
            threshold_candidates_output_path=(
                threshold_candidate_cache_path
            ),
            threshold_candidate_cache_context=(
                threshold_candidate_cache_context
            ),
            verbose=expcfg.INTERNAL_CALIBRATION_VERBOSE,
        )
    )
threshold_candidates = threshold_reduction[
    "threshold_candidates"
]
selected_policy_metrics = threshold_reduction[
    "selected_policy_metrics"
]
selected_thresholds = threshold_reduction[
    "selected_thresholds"
]
threshold_audit_summary = threshold_reduction[
    "threshold_audit_summary"
]
threshold_audit_path = threshold_reduction[
    "threshold_audit_path"
]

candidate_object_thresholds = pd.to_numeric(
    threshold_candidates.loc[
        threshold_candidates["decision_scope"].eq(
            "pixel_to_object"
        )
        & threshold_candidates["vote_threshold"].notna(),
        "vote_threshold",
    ],
    errors="raise",
).dropna().to_numpy(dtype=float)
expected_object_thresholds = np.asarray(
    expcfg.INTERNAL_CALIBRATION_OBJECT_THRESHOLDS,
    dtype=float,
)
candidate_threshold_matches = np.isclose(
    candidate_object_thresholds[:, None],
    expected_object_thresholds[None, :],
    rtol=0.0,
    atol=1e-7,
)
if (
    not candidate_threshold_matches.any(axis=0).all()
    or not candidate_threshold_matches.any(axis=1).all()
):
    raise RuntimeError(
        "The complete 0.75/0.80 object-threshold grid was not evaluated: "
        f"observed={sorted(set(candidate_object_thresholds.tolist()))}"
    )

selected_object_thresholds = pd.to_numeric(
    selected_thresholds.loc[
        selected_thresholds[
            "decision_scope"
        ].eq("pixel_to_object")
        & selected_thresholds["vote_threshold"].notna(),
        "vote_threshold",
    ],
    errors="raise",
).dropna().to_numpy(dtype=float)
if selected_object_thresholds.size:
    selected_threshold_matches = np.isclose(
        selected_object_thresholds[:, None],
        expected_object_thresholds[None, :],
        rtol=0.0,
        atol=1e-7,
    )
    if not selected_threshold_matches.any(axis=1).all():
        raise RuntimeError(
            "An unexpected 2-way object threshold was selected."
        )

model_metrics = build_model_metrics(
    selected_policy_metrics
)

save_parquet(
    selected_thresholds,
    output_paths["selected_thresholds"],
)
save_parquet(
    model_metrics,
    output_paths["model_metrics"],
)

display(
    selected_thresholds.sort_values(
        [
            "model_id",
            "random_state",
            "decision_scope",
        ]
    ).head(expcfg.INTERNAL_CALIBRATION_MAX_ROWS_TO_DISPLAY)
)

[03B thresholds 1/180] runners=1 models=320
[03B thresholds 2/180] runners=1 models=320
[03B thresholds 3/180] runners=1 models=320
[03B thresholds 4/180] runners=1 models=320
[03B thresholds 5/180] runners=1 models=320
[03B thresholds 6/180] runners=1 models=320
[03B thresholds 7/180] runners=1 models=320
[03B thresholds 8/180] runners=1 models=320
[03B thresholds 9/180] runners=1 models=320
[03B thresholds 10/180] runners=1 models=320
[03B thresholds 11/180] runners=1 models=320
[03B thresholds 12/180] runners=1 models=320
[03B thresholds 13/180] runners=1 models=320
[03B thresholds 14/180] runners=1 models=320
[03B thresholds 15/180] runners=1 models=320
[03B thresholds 16/180] runners=1 models=320
[03B thresholds 17/180] runners=1 models=320
[03B thresholds 18/180] runners=1 models=320
[03B thresholds 19/180] runners=1 models=320
[03B thresholds 20/180] runners=1 models=320
[03B thresholds 21/180] runners=1 models=320
[03B thresholds 22/180] runners=1 models=320
[03B thresholds 23/

,model_id,random_state,decision_scope,lower_quantile,upper_quantile,vote_threshold,lower_threshold,upper_threshold
36849,model_000683027d85536001d4,0,direct,1.00,0.25,NaN,-0.012115,0.426306
35585,model_000a8a461ef15e54288c,0,direct,0.75,0.25,NaN,-0.205505,0.318645
30366,model_000c26121aaa44a9d4f0,0,pixel_to_object,1.00,0.25,NaN,0.473684,0.714286
23836,model_00147d781b3f1ff082c4,0,direct,NaN,NaN,NaN,0.000000,0.000000
24456,model_0017f2ddf28dfaa5ffc3,0,direct,0.90,0.00,NaN,-0.984631,0.085398
158,model_0019149678d1cfda389d,0,direct,0.75,0.00,NaN,-2.246266,0.049157
12985,model_001ed4e4eb4e02dc1f8e,0,direct,0.50,0.00,NaN,-0.909411,0.002160
12986,model_001ed4e4eb4e02dc1f8e,1,direct,0.50,0.00,NaN,-1.016177,0.011308
12987,model_001ed4e4eb4e02dc1f8e,2,direct,0.50,0.00,NaN,-0.886938,0.000093
13711,model_0020b15e2e07b79d316e,0,pixel_to_object,NaN,NaN,0.8,0.800000,0.800000


## 7 — Sélection des modèles

Les graines sont d’abord agrégées. La réduction de complexité sélectionne
ensuite le plus petit `k` dans le plateau FN/FP à `m` fixé, puis le plus petit
`m` dans le plateau restant. Le front de Pareto final est calculé sans poids.

In [9]:
(
    selected_models,
    selected_runs,
    model_selection_audit,
) = select_calibrated_models(
    model_catalog=model_catalog,
    candidate_runs=candidate_runs,
    selected_policy_metrics=selected_policy_metrics,
    model_metrics=model_metrics,
    rule_diagnostics=rule_diagnostics,
    expected_n_folds=(
        expcfg.INTERNAL_CALIBRATION_N_SPLITS
    ),
)

if selected_models.empty:
    raise RuntimeError(
        "No model survived internal calibration."
    )

selected_track_ids = set(
    selected_models.merge(
        model_catalog[
            ["model_id", "track_id"]
        ],
        on="model_id",
        how="left",
        validate="one_to_one",
    )["track_id"]
)
if selected_track_ids != expected_track_ids:
    raise RuntimeError(
        "Every track E1-E8 must retain at least one model."
    )

model_audit_summary = summarize_selection_audit(
    model_selection_audit,
    model_catalog,
)
selection_audit_summary = (
    pd.concat(
        [threshold_audit_summary, model_audit_summary],
        ignore_index=True,
        sort=False,
    )
    .groupby(
        [
            "selection_level",
            "track_id",
            "stage",
            "decision",
            "reason_code",
        ],
        as_index=False,
        sort=False,
        dropna=False,
    )[["n_rows", "n_models"]]
    .sum()
)
finalize_streamed_selection_audit(
    threshold_audit_path,
    model_selection_audit,
    output_paths["selection_audit"],
)

save_parquet(
    selected_models,
    output_paths["selected_models"],
)
save_parquet(
    selected_runs,
    output_paths["selected_runs"],
)

WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/hsi_nuts/results/03B_internal_calibration_8tracks_v5_px_qc_v1/selected_runs.parquet')

## 8 — Audit visuel de la sélection

Les graphiques suivants présentent les éliminations, leurs raisons et les
modèles conservés. Ils sont descriptifs : ils ne participent pas à la sélection.

In [10]:
eliminations = (
    selection_audit_summary.loc[
        selection_audit_summary["decision"].eq("eliminated")
    ]
    .groupby(
        ["track_id", "stage", "reason_code"],
        dropna=False,
        as_index=False,
    )
    .agg(n_eliminated=("n_models", "sum"))
)

if not eliminations.empty:
    fig = px.bar(
        eliminations,
        x="stage",
        y="n_eliminated",
        color="reason_code",
        facet_col="track_id",
        facet_col_wrap=4,
        title="Éliminations par étape, track et raison",
        labels={
            "stage": "Étape",
            "n_eliminated": "Modèles éliminés",
            "reason_code": "Raison",
        },
    )
    fig.update_layout(
        template="plotly_white",
        legend_title_text="Raison",
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

selected_counts = (
    selected_models.merge(
        model_catalog[
            ["model_id", "track_id", "evaluation_track"]
        ],
        on="model_id",
        how="left",
        validate="one_to_one",
    )
    .groupby(
        ["track_id", "evaluation_track"],
        as_index=False,
    )
    .agg(n_selected=("model_id", "nunique"))
)

fig = px.bar(
    selected_counts,
    x="track_id",
    y="n_selected",
    hover_data=["evaluation_track"],
    title="Modèles sélectionnés par track",
    labels={
        "track_id": "Track",
        "n_selected": "Modèles sélectionnés",
    },
)
fig.update_layout(
    template="plotly_white",
    showlegend=False,
)
fig.show()

threshold_summary = (
    selected_policy_metrics.merge(
        model_catalog[
            ["model_id", "track_id", "decision_mode"]
        ],
        on="model_id",
        how="left",
        validate="many_to_one",
    )
)

fig = px.scatter(
    threshold_summary,
    x="target_miss_rate",
    y="false_accept_rate",
    color="track_id",
    symbol="decision_scope",
    hover_data=[
        "model_id",
        "lower_quantile",
        "upper_quantile",
        "vote_threshold",
        "worst_target_miss_rate",
        "worst_false_accept_rate",
    ],
    title=(
        "Politiques de seuil retenues — "
        "faux négatifs versus faux positifs"
    ),
    labels={
        "target_miss_rate": "Taux de faux négatifs",
        "false_accept_rate": "Taux de faux positifs",
        "track_id": "Track",
        "decision_scope": "Portée",
    },
)
fig.add_vline(
    x=0.05,
    line_dash="dash",
    line_color="firebrick",
    annotation_text="limite FN moyenne",
)
fig.add_hline(
    y=0.30,
    line_dash="dot",
    line_color="darkorange",
    annotation_text="limite FP moyenne",
)
fig.update_layout(template="plotly_white")
fig.show()

In [11]:
object_threshold_audit = (
    threshold_summary.loc[
        threshold_summary["vote_threshold"].notna(),
        [
            "track_id",
            "model_id",
            "vote_threshold",
            "target_miss_rate",
            "false_accept_rate",
            "worst_target_miss_rate",
            "worst_false_accept_rate",
        ],
    ]
    .sort_values(
        ["track_id", "model_id", "vote_threshold"]
    )
)

if not object_threshold_audit.empty:
    fig = px.line(
        object_threshold_audit,
        x="vote_threshold",
        y="target_miss_rate",
        color="model_id",
        facet_col="track_id",
        markers=True,
        title=(
            "Effet des seuils objets 0.75 et 0.80 "
            "sur les faux négatifs"
        ),
        labels={
            "vote_threshold": "Seuil objet",
            "target_miss_rate": "Taux de faux négatifs",
            "model_id": "Modèle",
        },
    )
    fig.update_layout(
        template="plotly_white",
        showlegend=False,
    )
    fig.update_xaxes(
        tickmode="array",
        tickvals=[0.75, 0.80],
    )
    fig.show()

In [12]:
threshold_candidates_for_plot = (
    sample_threshold_candidates_for_plot(
        threshold_candidates,
        selected_policy_metrics,
        model_catalog,
        max_rows_per_track_scope=(
            expcfg
            .INTERNAL_CALIBRATION_MAX_PLOT_CANDIDATES_PER_TRACK_SCOPE
        ),
    )
)

for track_id in expcfg.SIMCA_EVALUATION_TRACK_IDS:
    track_models = model_catalog.loc[
        model_catalog["track_id"].eq(track_id)
    ]
    if track_models.empty:
        continue

    decision_mode = str(
        track_models["decision_mode"].iloc[0]
    )
    constraints = {
        rule["metric"]: float(rule["value"])
        for rule in (
            expcfg.INTERNAL_CALIBRATION_THRESHOLD_CONSTRAINTS[
                decision_mode
            ]
        )
    }
    scopes = ["direct"]
    if track_models["projection_level"].eq(
        "pixel_projection"
    ).all():
        scopes.append("pixel_to_object")

    for decision_scope in scopes:
        plot_threshold_tradeoff(
            threshold_candidates_for_plot,
            selected_policy_metrics,
            model_catalog,
            track_id=track_id,
            decision_scope=decision_scope,
            fn_limit=constraints.get("target_miss_rate"),
            fp_limit=constraints.get("false_accept_rate"),
        )

In [13]:
object_threshold_candidates = (
    threshold_candidates_for_plot.merge(
        model_catalog[["model_id", "track_id"]],
        on="model_id",
        how="left",
        validate="many_to_one",
    )
    .loc[
        lambda frame: frame["vote_threshold"].notna()
    ]
)

fig = px.box(
    object_threshold_candidates,
    x="vote_threshold",
    y="target_miss_rate",
    color="track_id",
    points="outliers",
    hover_data=[
        "model_id",
        "false_accept_rate",
        "worst_target_miss_rate",
        "worst_false_accept_rate",
    ],
    title=(
        "Effet des seuils objets 0,75 et 0,80 "
        "sur les faux négatifs"
    ),
    labels={
        "vote_threshold": "Seuil objet",
        "target_miss_rate": "Taux de faux négatifs",
        "track_id": "Track",
    },
)
fig.update_xaxes(
    tickmode="array",
    tickvals=list(
        expcfg.INTERNAL_CALIBRATION_OBJECT_THRESHOLDS
    ),
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(template="plotly_white")
fig.show()

## 9 — Matérialisation des seules prédictions sélectionnées

Les OOF de tous les candidats restent dans les checkpoints. Seules les
projections nécessaires aux modèles sélectionnés sont chargées et exportées.

In [14]:
selected_execution_domain = (
    selected_runs.merge(
        model_catalog[
            [
                "model_id",
                "projection_level",
            ]
        ],
        on="model_id",
        how="left",
        validate="many_to_one",
    )
)

oof_objects, oof_pixels = (
    load_selected_oof_predictions_from_checkpoint_8tracks(
        checkpoint_run_dir,
        selected_execution_domain,
    )
)

save_parquet(
    oof_objects,
    output_paths["oof_object_predictions"],
)
save_parquet(
    oof_pixels,
    output_paths["oof_pixel_predictions"],
)

WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/hsi_nuts/results/03B_internal_calibration_8tracks_v5_px_qc_v1/oof_pixel_predictions.parquet')

## 10 — Intégrité, absence de score et manifeste

In [15]:
canonical_outputs = {
    "track_contracts": track_contracts,
    "folds": calibration_folds,
    "fold_diagnostics": fold_diagnostics,
    "model_catalog": model_catalog,
    "candidate_runs": candidate_runs,
    "fit_diagnostics": fit_diagnostics,
    "rule_diagnostics": rule_diagnostics,
    "projection_shift": projection_shift,
    "threshold_metrics": pd.DataFrame(
        columns=(
            expcfg.INTERNAL_CALIBRATION_THRESHOLD_METRIC_COLUMNS
        )
    ),
    "model_metrics": model_metrics,
    "selected_models": selected_models,
    "selected_runs": selected_runs,
    "selected_thresholds": selected_thresholds,
    "selection_audit": pd.DataFrame(
        columns=(
            expcfg.INTERNAL_CALIBRATION_SELECTION_AUDIT_COLUMNS
        )
    ),
    "technical_events": technical_events,
    "oof_object_predictions": oof_objects,
    "oof_pixel_predictions": oof_pixels,
}

assert_no_forbidden_score_columns(
    canonical_outputs
)

for key, table in canonical_outputs.items():
    if key in output_paths and key not in {
        "checkpoint_manifest",
    }:
        expected_columns = list(table.columns)
        disk_columns = list(
            pq.ParquetFile(
                output_paths[key]
            ).schema_arrow.names
        )
        if disk_columns != expected_columns:
            raise RuntimeError(
                f"Schema changed after persistence for {key}."
            )

artifact_rows = []
for key, path in output_paths.items():
    if key == "checkpoint_manifest":
        continue
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() != ".parquet":
        continue

    parquet = pq.ParquetFile(path)
    artifact_rows.append(
        {
            "name": key,
            "path": str(path),
            "row_count": int(parquet.metadata.num_rows),
            "columns": list(parquet.schema_arrow.names),
            "sha256": sha256_file(path),
        }
    )

manifest = {
    "protocol_version": expcfg.PROTOCOL_VERSION,
    "schema_version": expcfg.RESULTS_SCHEMA_VERSION,
    "protocol_hash": protocol_hash,
    "execution_parent_protocol_hash": execution_protocol_hash,
    "protocol_amendment_scope": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_AMENDMENT_SCOPE
    ),
    "pca_selection_fingerprint": pca_selection_fingerprint,
    "track_contract_hash": track_contract_hash,
    "fold_contract_hash": fold_contract_hash,
    "configuration_hash": configuration_hash,
    "checkpoint_run_dir": str(checkpoint_run_dir),
    "selection_profile_id": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_PROFILE_ID
    ),
    "selection_parent_profile_id": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_PARENT_PROFILE_ID
    ),
    "selection_profile_sha256": selection_profile_hash,
    "selection_amendment_reason": (
        expcfg.INTERNAL_CALIBRATION_SELECTION_AMENDMENT_REASON
    ),
    "threshold_candidate_cache_context_sha256": sha256_payload(
        threshold_candidate_cache_context
    ),
    "object_thresholds": list(expcfg.INTERNAL_CALIBRATION_OBJECT_THRESHOLDS),
    "target_uncertain_policy": (
        expcfg.INTERNAL_CALIBRATION_TARGET_UNCERTAIN_POLICY
    ),
    "artifacts": artifact_rows,
}

output_paths["checkpoint_manifest"].write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

display(
    pd.DataFrame(artifact_rows)[
        ["name", "row_count", "sha256"]
    ]
)

print(
    f"03B {expcfg.PROTOCOL_VERSION} completed: "
    f"{len(model_catalog)} initial models, "
    f"{len(selected_models)} selected models."
)

,name,row_count,sha256
0,track_contracts,8,0c84048b15db83d172d61cbfb6c4c7708f5038aed05b90...
1,folds,209,b70d838a126116edcbe29316247a2db03437f3a06b7ceb...
2,fold_diagnostics,2,7a9f81b246d590315eea52c21abdec908450501f059510...
3,model_catalog,74240,a6e9b638bb66a94e9ed980195b2bc873d09e38fa5327e9...
4,candidate_runs,124160,c7a48cc99364f57107f9b8aff409c31594610e646d578c...
5,fit_diagnostics,5680,89b39cd14502ed06277d1c46d691238dec5cd67e47b794...
6,rule_diagnostics,124160,94a176dc71601a2e90735286620d7d084f92c1ea96fe63...
7,projection_shift,124160,d01a3420471522ac832627e81c1e34ebce7bc414453e9a...
8,oof_object_predictions,11077,ee4cc48ae2f94d9a4d07f7814e17a64458f55efbeeb5ac...
9,oof_pixel_predictions,344195,0bef7eb6220a6c387ba8e7f25fe98f2898cc64337c6979...


03B 8tracks_v5 completed: 74240 initial models, 39 selected models.
